In [1]:
import nltk

nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")

# For newer NLTK versions, you may also need:
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt to /home/phuong/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/phuong/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /home/phuong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/phuong/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [2]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
print("Processor loaded successfully!")

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",   # important
)
print("Model loaded successfully!")

/home/phuong/anaconda3/envs/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Processor loaded successfully!


/home/phuong/anaconda3/envs/vlm-llava/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()
Loading checkpoint shards: 100%|██████████| 3/3 [02:59<00:00, 59.92s/it]

Model loaded successfully!


In [3]:
from PIL import Image

def prepare_input(processor, img_path: str, prompt: str):
    image = Image.open(img_path).convert("RGB")

    def create_messages(img):
        return [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt},
                ],
            },
        ]

    inputs = processor.apply_chat_template(
        create_messages(image), 
        add_generation_prompt=True, 
        tokenize=True, 
        return_dict=True, 
        return_tensors="pt"
    )

    return inputs

In [4]:
def move_inputs_to_device(inputs, model):
    """
    Move processor outputs to the model device.

    Keeps integer tensors such as input_ids unchanged.
    Casts floating tensors such as pixel_values to model dtype.
    """
    device = next(model.parameters()).device
    model_dtype = getattr(model, "dtype", torch.float16)

    moved = {}

    for k, v in inputs.items():
        if torch.is_tensor(v):
            if torch.is_floating_point(v):
                moved[k] = v.to(device=device, dtype=model_dtype)
            else:
                moved[k] = v.to(device=device)
        else:
            moved[k] = v

    return moved

In [5]:
def get_image_token_id(model, tokenizer):
    """
    Get LLaVA image token id.
    """
    image_token_id = getattr(model.config, "image_token_index", None)

    if image_token_id is None:
        image_token_id = tokenizer.convert_tokens_to_ids("<image>")

    return image_token_id

In [6]:
def resolve_image_token_indices(
    input_ids,
    token_attn_key_len,
    current_step,
    model,
    tokenizer,
):
    """
    Resolve image-token positions in the attention key dimension.

    This handles two cases:

    Case 1:
        input_ids already contains many expanded image tokens.

    Case 2:
        input_ids contains one <image> placeholder, and LLaVA internally
        expands it into many visual patch tokens.

    Assumption:
        batch size = 1, single image input.
    """
    image_token_id = get_image_token_id(model, tokenizer)

    raw_img_positions = (
        input_ids[0] == image_token_id
    ).nonzero(as_tuple=False).flatten()

    if len(raw_img_positions) == 0:
        return torch.empty(0, dtype=torch.long)

    raw_prompt_len = input_ids.shape[1]

    # During decoding:
    # key_len = expanded_prompt_len + number_of_generated_tokens_seen
    #
    # At step 0, after feeding the first generated token:
    # key_len = expanded_prompt_len + 1
    expanded_prompt_len = token_attn_key_len - (current_step + 1)

    # Case 1: image tokens are already expanded in input_ids.
    if len(raw_img_positions) > 1:
        return raw_img_positions.cpu()

    # Case 2: one placeholder expanded internally.
    placeholder_pos = int(raw_img_positions[0].item())

    # One placeholder token is replaced by num_image_tokens visual tokens.
    num_image_tokens = expanded_prompt_len - (raw_prompt_len - 1)

    if num_image_tokens <= 0:
        return torch.empty(0, dtype=torch.long)

    image_start = placeholder_pos
    image_end = placeholder_pos + num_image_tokens

    return torch.arange(image_start, image_end, dtype=torch.long)

In [7]:
def is_sentence_end_token(
    token_id,
    tokenizer,
    eos_token_id=None,
    treat_newline_as_boundary=True,
):
    """
    Check whether a token is sentence-ending, newline boundary, or EOS.
    """
    token_id = int(token_id)

    if eos_token_id is not None and token_id == eos_token_id:
        return True

    # Original LLaMA/LLaVA dot ids commonly seen in older code.
    if token_id in {869, 29889}:
        return True

    token_text = tokenizer.decode(
        [token_id],
        skip_special_tokens=False,
    )

    # Important: check newline BEFORE .strip(),
    # because "\n".strip() becomes "".
    if treat_newline_as_boundary and ("\n" in token_text or "\r" in token_text):
        return True

    stripped = token_text.strip()

    if stripped in {".", "!", "?", "。", "！", "？"}:
        return True

    return False

In [8]:
def build_candidate_record(token_ids, probs, tokenizer):
    """
    Convert candidate token ids/probs into readable records.
    """
    records = []

    for token_id, prob in zip(token_ids, probs):
        token_id = int(token_id)
        prob = float(prob)

        records.append(
            {
                "token_id": token_id,
                "token_text": tokenizer.decode(
                    [token_id],
                    skip_special_tokens=False,
                ),
                "prob": prob,
            }
        )

    return records

In [9]:
def select_token_from_logits(logits, temperature=0.0):
    """
    Select next token from logits.

    temperature <= 0:
        greedy decoding

    temperature > 0:
        multinomial sampling
    """
    if temperature is None or temperature <= 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [10]:
def normalize_prefix_ids(prefix_ids, device=None):
    """
    Convert prefix_ids into a Python list[int].

    prefix_ids can be:
        None
        list[int]
        torch.Tensor shape [T]
        torch.Tensor shape [1, T]
    """
    if prefix_ids is None:
        return []

    if torch.is_tensor(prefix_ids):
        prefix_ids = prefix_ids.detach().cpu()

        if prefix_ids.dim() == 2:
            prefix_ids = prefix_ids[0]

        return [int(x) for x in prefix_ids.tolist()]

    return [int(x) for x in prefix_ids]


In [11]:
def append_generated_ids_to_inputs(inputs, generated_ids):
    """
    Append generated token ids to prepared multimodal inputs.

    This creates:
        original prompt + generated prefix

    The image tensors are kept unchanged.
    """
    generated_ids = normalize_prefix_ids(generated_ids)

    out = {}

    for k, v in inputs.items():
        if torch.is_tensor(v):
            out[k] = v.clone()
        else:
            out[k] = v

    if len(generated_ids) == 0:
        return out

    device = out["input_ids"].device
    dtype = out["input_ids"].dtype

    gen_tensor = torch.tensor(
        [generated_ids],
        dtype=dtype,
        device=device,
    )

    out["input_ids"] = torch.cat(
        [out["input_ids"], gen_tensor],
        dim=-1,
    )

    if "attention_mask" in out and out["attention_mask"] is not None:
        gen_mask = torch.ones(
            (out["attention_mask"].shape[0], len(generated_ids)),
            dtype=out["attention_mask"].dtype,
            device=out["attention_mask"].device,
        )

        out["attention_mask"] = torch.cat(
            [out["attention_mask"], gen_mask],
            dim=-1,
        )

    return out

In [12]:
import re
import torch

@torch.inference_mode()
def generate_sentence_with_attn(
    model,
    processor,
    inputs,
    prefix_ids=None,
    max_new_tokens=64,
    min_new_tokens=3,
    temperature=0.0,
    stop_at_sentence_end=True,

    # Attention options
    selected_layers=None,
    keep_attn_on_cpu=True,
    return_full_attn=False,

    # Uncertainty/checkpoint options
    top_k=5,
    accumulate_prob=0.5,
    enable_uncertainty_check=True,
    checkpoint_once=True,
    stop_if_sentence_end_in_candidates=True,

    # Candidate branch testing
    force_first_token_id=None,

    debug=False,
):
    """
    Generate a sentence from prepared LLaVA-HF inputs, optionally continuing
    from a generated prefix.

    Args:
        prefix_ids:
            Previously generated token ids to append to the original prompt
            before generation starts. This is the replacement for the old
            generate_sentence_with_prefix(..., prefix=...).

        force_first_token_id:
            If not None, force this token as the first newly decoded token.
            Used when testing PHG candidate branches.

    Returns:
        Same schema as before, plus:
            prefix_ids:
                Prefix used for this generation call.

            full_generated_ids:
                prefix_ids + generated_ids.

            full_generated_text:
                Decoded text of prefix_ids + generated_ids.
    """

    def dprint(*args):
        if debug:
            print(*args)

    model.eval()

    tokenizer = processor.tokenizer
    device = next(model.parameters()).device

    prefix_ids = normalize_prefix_ids(prefix_ids)

    # Move base inputs first, then append prefix on the same device.
    base_inputs = move_inputs_to_device(inputs, model)

    working_inputs = append_generated_ids_to_inputs(
        base_inputs,
        prefix_ids,
    )

    input_ids = working_inputs["input_ids"]

    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None:
        eos_token_id = getattr(model.config, "eos_token_id", None)

    attention_mask = working_inputs.get("attention_mask", None)

    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids, device=input_ids.device)

    # -------------------------------------------------------
    # 1. Prefill pass: prompt + prefix + image.
    # -------------------------------------------------------
    prefill_outputs = model(
        **working_inputs,
        use_cache=True,
        output_attentions=False,
        return_dict=True,
    )

    past_key_values = prefill_outputs.past_key_values
    logits_for_next = prefill_outputs.logits[:, -1, :]

    # -------------------------------------------------------
    # 2. Storage.
    # -------------------------------------------------------
    generated_ids = []
    steps = []

    image_token_indices = None

    checkpointed = False
    checkpoint_input_ids = None
    checkpoint_generated_ids = None
    checkpoint_text = None

    candidates = None
    candidate_records = []

    has_uncertainty = False
    first_uncertain_step = None
    uncertain_steps = []

    stop_reason = None

    # -------------------------------------------------------
    # 3. Decoding loop.
    # -------------------------------------------------------
    for step in range(max_new_tokens):
        probs_for_next = torch.softmax(logits_for_next, dim=-1)
        max_prob = float(torch.max(probs_for_next).item())

        # If we force the first token for candidate branch testing,
        # do not let uncertainty logic override that forced token.
        forced_first_step = (
            force_first_token_id is not None
            and step == 0
        )

        is_low_confidence = (
            enable_uncertainty_check
            and not forced_first_step
            and max_prob < accumulate_prob
        )

        if is_low_confidence:
            has_uncertainty = True
            uncertain_steps.append(step)

            if first_uncertain_step is None:
                first_uncertain_step = step

        force_stop_token = None

        # ---------------------------------------------------
        # 3.1 Uncertainty path logic.
        # ---------------------------------------------------
        should_store_uncertainty = (
            is_low_confidence
            and (not checkpoint_once or not checkpointed)
        )

        if should_store_uncertainty:
            top_k_probs, top_k_indices = torch.topk(
                probs_for_next,
                k=top_k,
                dim=-1,
            )

            cumsum_probs = torch.cumsum(top_k_probs, dim=-1)

            cumsum_ids = (
                cumsum_probs >= accumulate_prob
            ).nonzero(as_tuple=True)[1]

            if len(cumsum_ids) > 0:
                min_k = int(cumsum_ids[0].item()) + 1
            else:
                min_k = top_k

            selected_candidate_ids = (
                top_k_indices[0, :min_k]
                .detach()
                .cpu()
                .tolist()
            )

            selected_candidate_probs = (
                top_k_probs[0, :min_k]
                .detach()
                .cpu()
                .tolist()
            )

            current_candidate_record = {
                "step": step,
                "max_prob": max_prob,
                "threshold": accumulate_prob,
                "candidates": build_candidate_record(
                    selected_candidate_ids,
                    selected_candidate_probs,
                    tokenizer,
                ),
            }

            candidate_records.append(current_candidate_record)

            if candidates is None:
                candidates = current_candidate_record["candidates"]

            # Store checkpoint before appending the uncertain token.
            if not checkpointed:
                checkpointed = True

                # Checkpoint in terms of generated text only:
                # previous prefix + tokens generated so far.
                checkpoint_generated_list = prefix_ids + generated_ids

                checkpoint_generated_ids = torch.tensor(
                    [checkpoint_generated_list],
                    dtype=input_ids.dtype,
                    device=input_ids.device,
                )

                # Checkpoint in terms of full model input:
                # original prompt + generated checkpoint.
                checkpoint_input_ids = append_generated_ids_to_inputs(
                    base_inputs,
                    checkpoint_generated_list,
                )["input_ids"]

                checkpoint_text = tokenizer.decode(
                    checkpoint_generated_list,
                    skip_special_tokens=True,
                ).strip()

            dprint(f"[uncertainty path] step={step}, max_prob={max_prob:.4f}")
            dprint("candidates:", current_candidate_record["candidates"])

            # Stop if sentence-end or EOS is inside candidate set.
            if stop_if_sentence_end_in_candidates:
                for cand_id in selected_candidate_ids:
                    if is_sentence_end_token(
                        cand_id,
                        tokenizer,
                        eos_token_id=eos_token_id,
                        treat_newline_as_boundary=True,
                    ):
                        force_stop_token = int(cand_id)
                        stop_reason = "sentence_end_or_eos_in_candidates"
                        break

        # ---------------------------------------------------
        # 3.2 Choose next token.
        # ---------------------------------------------------
        if force_stop_token is not None:
            next_token = torch.tensor(
                [[force_stop_token]],
                dtype=torch.long,
                device=device,
            )

        elif forced_first_step:
            next_token = torch.tensor(
                [[int(force_first_token_id)]],
                dtype=torch.long,
                device=device,
            )

        else:
            next_token = select_token_from_logits(
                logits_for_next,
                temperature=temperature,
            )

        token_id = int(next_token[0, 0].item())

        selected_token_prob = float(
            probs_for_next[0, token_id].item()
        )

        token_text = tokenizer.decode(
            [token_id],
            skip_special_tokens=False,
        )

        # ---------------------------------------------------
        # 3.3 Feed chosen token back into model.
        # This gives attention OF the generated token.
        # ---------------------------------------------------
        new_attention = torch.ones(
            (attention_mask.shape[0], 1),
            dtype=attention_mask.dtype,
            device=attention_mask.device,
        )

        attention_mask = torch.cat(
            [attention_mask, new_attention],
            dim=-1,
        )

        step_outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
            output_attentions=True,
            return_dict=True,
        )

        generated_ids.append(token_id)

        # ---------------------------------------------------
        # 3.4 PHG-friendly attention extraction.
        # ---------------------------------------------------
        num_layers = len(step_outputs.attentions)

        if selected_layers is None:
            layer_ids = list(range(num_layers))
        else:
            layer_ids = []

            for layer_id in selected_layers:
                if layer_id < 0:
                    layer_id = num_layers + layer_id

                layer_ids.append(layer_id)

        attn_by_layer = {}
        image_attn_by_layer = {}

        for layer_id in layer_ids:
            attn = step_outputs.attentions[layer_id]

            token_attn = attn[0, :, -1, :].detach()

            if image_token_indices is None:
                image_token_indices = resolve_image_token_indices(
                    input_ids=input_ids,
                    token_attn_key_len=token_attn.shape[-1],
                    current_step=step,
                    model=model,
                    tokenizer=tokenizer,
                )

            valid_img_idx = image_token_indices[
                image_token_indices < token_attn.shape[-1]
            ]

            if keep_attn_on_cpu:
                token_attn_for_store = token_attn.float().cpu()
            else:
                token_attn_for_store = token_attn

            if return_full_attn:
                attn_by_layer[layer_id] = token_attn_for_store

            if len(valid_img_idx) > 0:
                if keep_attn_on_cpu:
                    img_idx = valid_img_idx.cpu()
                    image_attn = token_attn_for_store[:, img_idx]
                else:
                    img_idx = valid_img_idx.to(token_attn.device)
                    image_attn = token_attn[:, img_idx]

                image_attn_by_layer[layer_id] = image_attn

        steps.append(
            {
                "step": step,
                "token_id": token_id,
                "token_text": token_text,
                "max_prob": max_prob,
                "selected_token_prob": selected_token_prob,
                "is_low_confidence": is_low_confidence,
                "attn_by_layer": attn_by_layer if return_full_attn else None,
                "image_attn_by_layer": image_attn_by_layer,
            }
        )

        # ---------------------------------------------------
        # 3.5 Update cache and logits for next step.
        # ---------------------------------------------------
        past_key_values = step_outputs.past_key_values
        logits_for_next = step_outputs.logits[:, -1, :]

        # ---------------------------------------------------
        # 3.6 Stop conditions.
        # ---------------------------------------------------
        if stop_reason is not None:
            break

        if eos_token_id is not None and token_id == eos_token_id:
            stop_reason = "eos_generated"
            break

        decoded_text_raw = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        )
        
        if (
            stop_at_sentence_end
            and len(generated_ids) >= min_new_tokens
            and re.search(r"([.!?。！？]\s*|\n+|\r+)$", decoded_text_raw)
        ):
            stop_reason = "sentence_end_generated"
            break

    # -------------------------------------------------------
    # 4. Final path decision.
    # -------------------------------------------------------
    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    full_generated_ids = prefix_ids + generated_ids

    full_generated_text = tokenizer.decode(
        full_generated_ids,
        skip_special_tokens=True,
    ).strip()

    if image_token_indices is None:
        image_token_indices = torch.empty(0, dtype=torch.long)

    if stop_reason is None:
        stop_reason = "max_new_tokens_reached"

    confidence_path = "uncertainty" if has_uncertainty else "certainty"

    return {
        "generated_text": generated_text,
        "generated_ids": generated_ids,

        "prefix_ids": prefix_ids,
        "full_generated_ids": full_generated_ids,
        "full_generated_text": full_generated_text,

        "confidence_path": confidence_path,
        "has_uncertainty": has_uncertainty,
        "first_uncertain_step": first_uncertain_step,
        "uncertain_steps": uncertain_steps,

        "stop_reason": stop_reason,

        "steps": steps,
        "image_token_indices": image_token_indices.cpu(),

        "checkpoint_input_ids": (checkpoint_input_ids.detach().cpu() if checkpoint_input_ids is not None else None),
        "checkpoint_generated_ids": (checkpoint_generated_ids.detach().cpu() if checkpoint_generated_ids is not None else None),
        "checkpoint_text": checkpoint_text,
        "candidates": candidates,
        "candidate_records": candidate_records,
    }

In [13]:
from scipy.ndimage import label, binary_closing
from typing import List, Dict, Optional, Tuple

def spatial_entropy(attn_map_2d: torch.Tensor, threshold: float = 1e-3) -> Dict:
    """
    Compute component-level spatial entropy for a 2D image-attention map.

    Supports both square and rectangular visual-token grids.
    """
    S = attn_map_2d.float()
    S = (S - S.min()) / (S.max() - S.min() + 1e-8)

    mean_val = torch.mean(S)
    activated = torch.relu(S - mean_val * 2.0)

    activated_np = (
        activated
        .detach()
        .cpu()
        .to(torch.float32)
        .numpy()
    )

    binary = (activated_np > threshold).astype(np.int32)

    labeled, num = label(
        binary,
        structure=np.ones((3, 3), dtype=np.int32),
    )

    total = float(activated.sum().item())

    if total <= 0:
        return {
            "spatial_entropy": float("inf"),
            "labeled_array": labeled,
            "num_components": 0,
        }

    probs = []

    for i in range(1, num + 1):
        comp_sum = activated_np[labeled == i].sum()

        if comp_sum > 0:
            probs.append(comp_sum / total)

    se = -sum(p * np.log(p) for p in probs if p > 0) if probs else 0.0

    return {
        "spatial_entropy": float(se),
        "labeled_array": labeled,
        "num_components": int(num),
    }

In [14]:
def remove_singletons(mask_bool):
    structure = np.ones((3, 3), dtype=bool)

    lab, _ = label(
        mask_bool.astype(bool),
        structure=structure,
    )

    counts = np.bincount(lab.ravel())

    keep = np.zeros_like(counts, dtype=bool)
    keep[1:] = counts[1:] >= 2

    return keep[lab]

In [15]:
import numpy as np

def compute_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool)
    b = b.astype(bool)

    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    return float(inter) / float(union + 1e-8)

In [16]:
def mask_is_compatible(
    new_mask: Optional[np.ndarray],
    masks_list: List[np.ndarray],
    iou_thresh: float = 0.5,
) -> bool:
    if new_mask is None:
        return False

    area = int(new_mask.astype(bool).sum())

    if area == 0:
        return False

    for old in masks_list:
        if old is None:
            continue

        iou = compute_iou(new_mask, old)

        if iou > iou_thresh:
            return False

    return True

In [17]:
def resolve_image_grid_shape(
    num_image_tokens: int,
    image_grid_shape: Optional[Tuple[int, int]] = None,
    inputs: Optional[Dict] = None,
) -> Tuple[int, int]:
    """
    Resolve image grid shape.

    Priority:
        1. Explicit image_grid_shape=(grid_h, grid_w)
        2. inputs["image_grid_thw"], if available
        3. Square fallback, e.g. LLaVA-1.5 24x24
    """
    if image_grid_shape is not None:
        grid_h, grid_w = image_grid_shape

        if grid_h * grid_w != num_image_tokens:
            raise ValueError(
                f"image_grid_shape={image_grid_shape} gives "
                f"{grid_h * grid_w} tokens, but attention has "
                f"{num_image_tokens} image tokens."
            )

        return int(grid_h), int(grid_w)

    if inputs is not None and "image_grid_thw" in inputs:
        grid_thw = inputs["image_grid_thw"]

        if torch.is_tensor(grid_thw):
            grid_thw = grid_thw.detach().cpu()

        t, h, w = grid_thw[0].tolist()

        if int(t) != 1:
            raise ValueError(
                f"Video/multi-frame grid detected: T={t}, H={h}, W={w}. "
                "This PHG code expects a single image."
            )

        if int(h) * int(w) != num_image_tokens:
            raise ValueError(
                f"image_grid_thw gives H*W={int(h) * int(w)}, "
                f"but attention has {num_image_tokens} image tokens."
            )

        return int(h), int(w)

    side = int(num_image_tokens ** 0.5)

    if side * side == num_image_tokens:
        return side, side

    raise ValueError(
        f"Cannot infer image grid from {num_image_tokens} image tokens. "
        "Pass image_grid_shape=(grid_h, grid_w)."
    )

In [18]:
def image_attn_to_grid(
    image_attn_1d: torch.Tensor,
    image_grid_shape: Optional[Tuple[int, int]] = None,
    inputs: Optional[Dict] = None,
) -> torch.Tensor:
    num_image_tokens = int(image_attn_1d.numel())

    grid_h, grid_w = resolve_image_grid_shape(
        num_image_tokens=num_image_tokens,
        image_grid_shape=image_grid_shape,
        inputs=inputs,
    )

    return image_attn_1d.reshape(grid_h, grid_w)

In [19]:
def get_kept_lh_from_step(
    step: Dict,
    image_grid_shape: Optional[Tuple[int, int]] = None,
    inputs: Optional[Dict] = None,
    attn_sum_threshold: float = 0.49,
    bottom_row_threshold: float = 0.05,
    min_layer: int = 1,
) -> List[Dict]:
    image_attn_by_layer = step["image_attn_by_layer"]

    results = []

    for layer_id, image_attn in image_attn_by_layer.items():
        image_attn = image_attn.detach().float().cpu()

        num_heads = image_attn.shape[0]

        for head_id in range(num_heads):
            attn_1d = image_attn[head_id]
            attn_sum = float(attn_1d.sum().item())

            if attn_sum < attn_sum_threshold:
                se = float("inf")
                bottom_row_focus = False
                n_comp = 0

            else:
                a2d = image_attn_to_grid(
                    attn_1d,
                    image_grid_shape=image_grid_shape,
                    inputs=inputs,
                )

                se_res = spatial_entropy(
                    a2d,
                    threshold=1e-3,
                )

                bottom_row_focus = bool(
                    (a2d.shape[0] > 0)
                    and (a2d[-1, :] > bottom_row_threshold).any()
                )

                se = float(se_res["spatial_entropy"])
                n_comp = int(se_res["num_components"])

            results.append(
                {
                    "layer": int(layer_id),
                    "head": int(head_id),
                    "attn_sum": attn_sum,
                    "spatial_entropy": se,
                    "bottom_row_focus": bottom_row_focus,
                    "num_components": n_comp,
                }
            )

    kept = [
        r
        for r in results
        if np.isfinite(r["spatial_entropy"])
        and r["attn_sum"] >= attn_sum_threshold
        and not r["bottom_row_focus"]
        and r["layer"] > min_layer
    ]

    if len(kept) < 1:
        by_sum = sorted(
            results,
            key=lambda x: x["attn_sum"],
            reverse=True,
        )

        kept = [
            x
            for x in by_sum
            if not x["bottom_row_focus"]
        ][:1]

    kept.sort(key=lambda x: x["spatial_entropy"])

    return kept

In [20]:
def get_object_mask_from_step(
    step: Dict,
    image_grid_shape: Optional[Tuple[int, int]] = None,
    inputs: Optional[Dict] = None,
    top_n_heads: int = 5,
    attn_sum_threshold: float = 0.49,
) -> Optional[np.ndarray]:
    kept = get_kept_lh_from_step(
        step,
        image_grid_shape=image_grid_shape,
        inputs=inputs,
        attn_sum_threshold=attn_sum_threshold,
    )

    if len(kept) < 1:
        return None

    binaries = []

    image_attn_by_layer = step["image_attn_by_layer"]

    for r in kept[:top_n_heads]:
        layer_id = r["layer"]
        head_id = r["head"]

        image_attn = image_attn_by_layer[layer_id].detach().float().cpu()
        attn_1d = image_attn[head_id]

        a2d = image_attn_to_grid(
            attn_1d,
            image_grid_shape=image_grid_shape,
            inputs=inputs,
        ).numpy()

        a2d = (a2d - a2d.min()) / (a2d.max() - a2d.min() + 1e-8)

        mean_val = np.mean(a2d)
        activated = np.maximum(a2d - mean_val * 2.0, 0)

        binary = (activated > 1e-8).astype(np.int32)
        binary = remove_singletons(binary)

        binary = binary_closing(
            binary,
            structure=np.ones((3, 3)),
        ).astype(np.int32)

        binaries.append(binary)

    if len(binaries) == 0:
        return None

    mask = np.median(
        np.stack(binaries, axis=0),
        axis=0,
    )

    return (mask > 0).astype(np.uint8)

In [21]:
try:
    import nltk
except ImportError:
    nltk = None


SHELL_NOUNS = {
    "variety", "kind", "type", "sort", "form", "category", "class", "genre",
    "subtype", "subset", "group", "set", "series", "sequence", "suite",
    "lineup", "selection", "array", "collection", "assortment", "mix",
    "blend", "combination", "mixture", "package", "bundle", "batch",
    "bunch", "cluster", "stack", "pile", "heap", "portfolio", "inventory",
    "list", "range", "spectrum", "continuum", "aggregation", "pool", "bucket"
}

GENERIC_BUCKETS = {
    "entity", "entities", "object", "objects", "thing", "things", "item",
    "items", "unit", "units", "component", "components", "element",
    "elements", "material", "materials", "content", "contents", "product",
    "products", "article", "articles", "asset", "assets", "resource",
    "resources", "ingredient", "ingredients", "stuff", "substance",
    "substances", "artifact", "artifacts", "entry", "entries", "record",
    "records", "row", "area", "space", "place", "location", "spot", "section", "part"
}

MEASURE_NOUNS = {
    "amount", "number", "quantity", "volume", "mass", "weight", "size",
    "degree", "level", "rate", "proportion", "percentage", "share", "ratio",
    "count", "total", "sum", "average", "mean", "median", "portion", "part",
    "piece", "section", "segment", "subset", "member", "instance", "sample",
    "example", "case", "occurrence", "pair", "couple", "trio", "dozen",
    "hundred", "thousand", "million"
}

IMAGE_DESCRIPTION_NOUNS = {
    "image", "photo", "picture", "scene", "view", "frame", "snapshot",
    "visual", "portrait", "landscape", "depiction", "atmosphere",
    "illustration", "rendering", "capture"
}

DIRECTIONAL_NOUNS = {
    "top", "bottom", "middle", "center", "left", "right", "side", "corner",
    "edge", "border", "margin", "foreground", "background", "midground",
    "front", "back", "rear", "frontside", "backside", "surface", "north",
    "south", "east", "west", "northeast", "northwest", "southeast",
    "southwest", "toward", "towards", "nearby", "near", "around", "across", "behind",
    "above", "below", "under", "over", "beside", "between"
}

OUTLIER_NOUNS = (
    SHELL_NOUNS
    .union(GENERIC_BUCKETS)
    .union(MEASURE_NOUNS)
    .union(IMAGE_DESCRIPTION_NOUNS)
    .union(DIRECTIONAL_NOUNS)
)


def detect_nouns(text: str, joiner: str = " ") -> List[str]:
    if nltk is None:
        raise ImportError("Please install nltk first: pip install nltk")

    tokens = nltk.word_tokenize(text)
    tags = nltk.pos_tag(tokens)

    keep = {"NN", "NNS", "NNP", "NNPS"}

    merged = []
    i = 0

    while i < len(tokens):
        if tags[i][1] in keep:
            j = i + 1
            phrase = [tokens[i]]

            while j < len(tokens) and tags[j][1] in keep:
                phrase.append(tokens[j])
                j += 1

            merged.append(joiner.join(phrase))
            i = j

        else:
            i += 1

    return merged


def find_sublist_start(a: List[int], b: List[int]) -> Optional[int]:
    if not b:
        return None

    n = len(a)
    m = len(b)

    for i in range(n - m + 1):
        if a[i : i + m] == b:
            return i

    return None


def find_noun_token_start(tokenizer, token_ids: List[int], noun: str) -> Optional[int]:
    variants = [
        noun,
        " " + noun,
    ]

    for text in variants:
        noun_ids = tokenizer.encode(
            text,
            add_special_tokens=False,
        )

        start = find_sublist_start(
            token_ids,
            noun_ids,
        )

        if start is not None:
            return start

    return None

In [22]:
def compute_ads_from_attention_map(
    attn_map_2d: torch.Tensor,
    foreground_ratio: float = 0.10,
    eps: float = 1e-8,
) -> float:
    """
    Compute ADS-style attention dispersion score.

    Lower ADS means attention is compact.
    Higher ADS means attention is diffuse.

    ADS(A) = (1 - m_foreground) * H_background
    """
    A = attn_map_2d.detach().float().cpu()
    A = torch.clamp(A, min=0)

    flat = A.flatten()
    total = flat.sum()

    if float(total.item()) <= eps:
        return float("inf")

    probs = flat / (total + eps)

    n = probs.numel()
    k = max(1, int(np.ceil(foreground_ratio * n)))

    _, top_idx = torch.topk(probs, k=k)

    foreground_mask = torch.zeros_like(probs, dtype=torch.bool)
    foreground_mask[top_idx] = True

    background_mask = ~foreground_mask

    m_foreground = float(probs[foreground_mask].sum().item())

    bg_probs = probs[background_mask]

    if bg_probs.numel() == 0 or float(bg_probs.sum().item()) <= eps:
        h_background = 0.0
    else:
        bg_probs = bg_probs / (bg_probs.sum() + eps)
        h = -torch.sum(bg_probs * torch.log(bg_probs + eps))
        h_background = float(h.item()) / float(np.log(max(n, 2)))

    ads = (1.0 - m_foreground) * h_background

    return float(ads)

In [23]:
def compute_ads_from_step(
    step: Dict,
    image_grid_shape=None,
    inputs=None,
    foreground_ratio: float = 0.10,
    top_n_heads: int = 3,
    attn_sum_threshold: float = 0.49,
) -> float:
    """
    Compute ADS from the most informative selected layer-head maps.

    Uses get_kept_lh_from_step(...) to select compact, image-attending heads.
    Returns the minimum ADS among the selected heads.
    """
    kept = get_kept_lh_from_step(
        step,
        image_grid_shape=image_grid_shape,
        inputs=inputs,
        attn_sum_threshold=attn_sum_threshold,
    )

    if len(kept) < 1:
        return float("inf")

    ads_values = []

    image_attn_by_layer = step["image_attn_by_layer"]

    for r in kept[:top_n_heads]:
        layer_id = r["layer"]
        head_id = r["head"]

        image_attn = image_attn_by_layer[layer_id].detach().float().cpu()
        attn_1d = image_attn[head_id]

        attn_2d = image_attn_to_grid(
            attn_1d,
            image_grid_shape=image_grid_shape,
            inputs=inputs,
        )

        ads = compute_ads_from_attention_map(
            attn_2d,
            foreground_ratio=foreground_ratio,
        )

        ads_values.append(ads)

    if len(ads_values) == 0:
        return float("inf")

    return float(min(ads_values))

In [24]:
def is_eos_stop(output, tokenizer) -> bool:
    """
    Check whether an output actually ended with EOS.
    """
    eos_token_id = tokenizer.eos_token_id

    if eos_token_id is None:
        return False

    ids = output.get("full_generated_ids", None)

    if not ids:
        ids = output.get("generated_ids", None)

    if not ids:
        return False

    return int(ids[-1]) == int(eos_token_id)

In [25]:
def is_sentence_boundary_stop(output, tokenizer) -> bool:
    """
    Check whether output reached a sentence boundary or EOS.
    """
    stop_reason = output.get("stop_reason", None)

    if stop_reason in {
        "sentence_end_generated",
        "sentence_end_or_eos_in_candidates",
        "eos_generated",
    }:
        return True

    return False

In [26]:
def extract_output_segment_by_abs_range(output, start_abs: int, end_abs: int):
    """
    Extract generated token segment and corresponding step records
    using absolute generated-token positions.

    output["prefix_ids"] are the tokens already present before this call.
    output["generated_ids"] are newly decoded tokens from this call.
    output["steps"] align with output["generated_ids"].
    """
    prefix_len = len(output.get("prefix_ids", []))

    rel_start = max(0, start_abs - prefix_len)
    rel_end = max(0, end_abs - prefix_len)

    segment_ids = output["generated_ids"][rel_start:rel_end]
    segment_steps = output["steps"][rel_start:rel_end]

    return segment_ids, segment_steps

In [27]:
def score_object_segment_for_memory(
    tokenizer,
    segment_token_ids: List[int],
    segment_steps: List[Dict],
    global_objects: List[str],
    global_masks: List[np.ndarray],
    sentence_objects: List[str],
    sentence_masks: List[np.ndarray],
    image_grid_shape=None,
    inputs=None,
    iou_thresh: float = 0.5,
    ads_thresh: float = 0.45,
    ads_foreground_ratio: float = 0.10,
    debug: bool = False,
):
    """
    Extract object mentions from a generated segment, build masks,
    check ADS compactness and IoU compatibility, then return accepted
    objects plus hallucination score.

    Memory used for compatibility:
        M_t = M_g union M_s
    """

    def dprint(*args):
        if debug:
            print(*args)

    segment_text = tokenizer.decode(
        segment_token_ids,
        skip_special_tokens=True,
    ).strip()

    if len(segment_token_ids) == 0 or len(segment_steps) == 0:
        return {
            "text": segment_text,
            "accepted_objects": [],
            "accepted_masks": [],
            "suspicious_objects": [],
            "hallucination_score": 0,
            "details": [],
        }

    try:
        nouns = set(detect_nouns(segment_text))
    except LookupError as e:
        raise LookupError(
            "NLTK data is missing. Run:\n"
            "import nltk\n"
            "nltk.download('punkt')\n"
            "nltk.download('averaged_perceptron_tagger')\n"
            "or for newer NLTK:\n"
            "nltk.download('averaged_perceptron_tagger_eng')"
        ) from e

    known_objects = set(global_objects).union(set(sentence_objects))

    effective_masks = list(global_masks) + list(sentence_masks)

    accepted_objects = []
    accepted_masks = []
    suspicious_objects = []
    details = []

    hallucination_score = 0

    for noun in nouns:
        noun_norm = noun.lower().strip()

        if noun_norm in OUTLIER_NOUNS:
            continue

        if noun_norm in known_objects:
            dprint(f"  [known object] {noun}")
            continue

        noun_start = find_noun_token_start(
            tokenizer,
            segment_token_ids,
            noun,
        )

        if noun_start is None:
            dprint(f"  [skip noun: cannot align] {repr(noun)}")
            continue

        if noun_start >= len(segment_steps):
            dprint(f"  [skip noun: no step] {repr(noun)} at {noun_start}")
            continue

        noun_step = segment_steps[noun_start]

        noun_mask = get_object_mask_from_step(
            noun_step,
            image_grid_shape=image_grid_shape,
            inputs=inputs,
            top_n_heads=5,
            attn_sum_threshold=0.49,
        )

        ads = compute_ads_from_step(
            noun_step,
            image_grid_shape=image_grid_shape,
            inputs=inputs,
            foreground_ratio=ads_foreground_ratio,
            top_n_heads=3,
            attn_sum_threshold=0.49,
        )

        has_valid_mask = (
            noun_mask is not None
            and int(noun_mask.astype(bool).sum()) > 0
        )

        compact_enough = (
            np.isfinite(ads)
            and ads <= ads_thresh
        )

        compatible = mask_is_compatible(
            noun_mask,
            effective_masks + accepted_masks,
            iou_thresh=iou_thresh,
        )

        accepted = bool(
            has_valid_mask
            and compact_enough
            and compatible
        )

        detail = {
            "noun": noun_norm,
            "token_start": noun_start,
            "ads": float(ads),
            "has_valid_mask": has_valid_mask,
            "compact_enough": compact_enough,
            "iou_compatible": compatible,
            "accepted": accepted,
        }

        details.append(detail)

        if accepted:
            dprint(f'  [grounded] "{noun}" ads={ads:.4f}')

            accepted_objects.append(noun_norm)
            accepted_masks.append(noun_mask)

            known_objects.add(noun_norm)

        else:
            hallucination_score += 1
            suspicious_objects.append(noun_norm)

            dprint(
                f'  [suspicious] "{noun}" '
                f"valid_mask={has_valid_mask}, "
                f"ads={ads:.4f}, "
                f"compact={compact_enough}, "
                f"compatible={compatible}"
            )

    return {
        "text": segment_text,
        "accepted_objects": accepted_objects,
        "accepted_masks": accepted_masks,
        "suspicious_objects": suspicious_objects,
        "hallucination_score": hallucination_score,
        "details": details,
    }

In [28]:
def generate_with_phg(
    model,
    processor,
    inputs,
    prefix_ids=None,
    iou_thresh: float = 0.5,
    ads_thresh: float = 0.45,
    ads_foreground_ratio: float = 0.10,
    max_rounds: int = 10,

    # Generation args
    max_new_tokens: int = 64,
    min_new_tokens: int = 3,
    temperature: float = 0.0,
    stop_at_sentence_end: bool = True,

    # Attention args
    selected_layers=None,

    # Uncertainty args
    top_k: int = 5,
    accumulate_prob: float = 0.5,

    # Grid args
    image_grid_shape: Optional[Tuple[int, int]] = None,

    debug: bool = False,
):
    """
    PHG generation wrapper following the uncertainty-checkpoint sentence
    decoding section.

    Main behavior:
        1. Decode normally while confidence is high.
        2. Confident completed sentences still update global object memory M_g.
        3. At uncertainty checkpoint, process the unprocessed unfinished prefix
           into temporary current-sentence memory M_s.
        4. Branch over candidate tokens.
        5. Score each candidate continuation using:
               mask validity
               ADS compactness
               IoU compatibility against M_g union M_s
        6. Select candidate by:
               (hallucination_score, candidate_rank)
        7. Add selected grounded objects to M_s.
        8. When sentence boundary is reached:
               M_g <- M_g union M_s
               M_s <- empty
    """

    def dprint(*args):
        if debug:
            print(*args)

    tokenizer = processor.tokenizer

    prefix_ids = normalize_prefix_ids(prefix_ids)

    accepted_generated_ids = prefix_ids.copy()

    # Global accepted object memory M_g
    global_objects = []
    global_masks = []

    # Temporary current-sentence memory M_s
    sentence_objects = []
    sentence_masks = []

    # Processed-prefix pointer pi_s.
    # Absolute index in accepted/generated answer-token space.
    processed_prefix_len = len(accepted_generated_ids)

    round_outputs = []
    decision_trace = []

    base_inputs = move_inputs_to_device(inputs, model)

    for round_idx in range(max_rounds):
        dprint(f"\n========== PHG ROUND {round_idx} ==========")
        dprint("[prefix]", tokenizer.decode(accepted_generated_ids, skip_special_tokens=True))
        dprint("[pi_s]", processed_prefix_len)

        current_output = generate_sentence_with_attn(
            model=model,
            processor=processor,
            inputs=base_inputs,
            prefix_ids=accepted_generated_ids,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            temperature=temperature,
            stop_at_sentence_end=stop_at_sentence_end,
            selected_layers=selected_layers,
            keep_attn_on_cpu=True,
            return_full_attn=False,
            top_k=top_k,
            accumulate_prob=accumulate_prob,
            enable_uncertainty_check=True,
            checkpoint_once=True,
            stop_if_sentence_end_in_candidates=True,
            force_first_token_id=None,
            debug=debug,
        )

        round_outputs.append(current_output)

        dprint("[generated]", repr(current_output["generated_text"]))
        dprint("[full]", repr(current_output["full_generated_text"]))
        dprint("[confidence_path]", current_output["confidence_path"])
        dprint("[stop_reason]", current_output["stop_reason"])

        # ===================================================
        # 1. Confident decoding path
        # ===================================================
        if current_output["confidence_path"] == "certainty":
            segment_ids = current_output["generated_ids"]
            segment_steps = current_output["steps"]

            segment_eval = score_object_segment_for_memory(
                tokenizer=tokenizer,
                segment_token_ids=segment_ids,
                segment_steps=segment_steps,
                global_objects=global_objects,
                global_masks=global_masks,
                sentence_objects=sentence_objects,
                sentence_masks=sentence_masks,
                image_grid_shape=image_grid_shape,
                inputs=base_inputs,
                iou_thresh=iou_thresh,
                ads_thresh=ads_thresh,
                ads_foreground_ratio=ads_foreground_ratio,
                debug=debug,
            )

            sentence_objects.extend(segment_eval["accepted_objects"])
            sentence_masks.extend(segment_eval["accepted_masks"])

            accepted_generated_ids = current_output["full_generated_ids"]
            processed_prefix_len = len(accepted_generated_ids)

            decision_trace.append(
                {
                    "round": round_idx,
                    "path": "certainty",
                    "stop_reason": current_output["stop_reason"],
                    "segment_eval": segment_eval,
                }
            )

            dprint("[accept certainty path]")
            dprint("[accepted objects in segment]", segment_eval["accepted_objects"])

            if is_sentence_boundary_stop(current_output, tokenizer):
                # Commit M_s -> M_g at sentence boundary.
                global_objects.extend(sentence_objects)
                global_masks.extend(sentence_masks)

                dprint("[commit sentence memory]")
                dprint("[M_g objects]", global_objects)

                sentence_objects = []
                sentence_masks = []
                processed_prefix_len = len(accepted_generated_ids)

                if is_eos_stop(current_output, tokenizer):
                    break

                # Continue to next sentence.
                continue

            # No sentence boundary yet. Continue decoding same unfinished sentence.
            continue

        # ===================================================
        # 2. Uncertainty-checkpoint path
        # ===================================================
        checkpoint_generated_tensor = current_output.get(
            "checkpoint_generated_ids",
            None,
        )

        if checkpoint_generated_tensor is None:
            checkpoint_generated_ids = accepted_generated_ids.copy()
        else:
            checkpoint_generated_ids = (
                checkpoint_generated_tensor[0]
                .detach()
                .cpu()
                .tolist()
            )

        checkpoint_pos = len(checkpoint_generated_ids)

        # ---------------------------------------------------
        # 2.1 Process unfinished prefix before checkpoint.
        #     Only process [pi_s, checkpoint_pos).
        # ---------------------------------------------------
        prefix_segment_ids, prefix_segment_steps = extract_output_segment_by_abs_range(
            output=current_output,
            start_abs=processed_prefix_len,
            end_abs=checkpoint_pos,
        )

        prefix_eval = score_object_segment_for_memory(
            tokenizer=tokenizer,
            segment_token_ids=prefix_segment_ids,
            segment_steps=prefix_segment_steps,
            global_objects=global_objects,
            global_masks=global_masks,
            sentence_objects=sentence_objects,
            sentence_masks=sentence_masks,
            image_grid_shape=image_grid_shape,
            inputs=base_inputs,
            iou_thresh=iou_thresh,
            ads_thresh=ads_thresh,
            ads_foreground_ratio=ads_foreground_ratio,
            debug=debug,
        )

        sentence_objects.extend(prefix_eval["accepted_objects"])
        sentence_masks.extend(prefix_eval["accepted_masks"])

        processed_prefix_len = checkpoint_pos

        dprint("[checkpoint_text]", repr(current_output.get("checkpoint_text", "")))
        dprint("[processed prefix text]", repr(prefix_eval["text"]))
        dprint("[prefix accepted objects]", prefix_eval["accepted_objects"])
        dprint("[pi_s updated]", processed_prefix_len)

        # ---------------------------------------------------
        # 2.2 If uncertainty chose sentence boundary from candidates,
        #     accept shortened sentence and commit M_s.
        # ---------------------------------------------------
        if current_output["stop_reason"] == "sentence_end_or_eos_in_candidates":
            accepted_generated_ids = current_output["full_generated_ids"]
            processed_prefix_len = len(accepted_generated_ids)

            global_objects.extend(sentence_objects)
            global_masks.extend(sentence_masks)

            decision_trace.append(
                {
                    "round": round_idx,
                    "path": "uncertainty_boundary",
                    "stop_reason": current_output["stop_reason"],
                    "prefix_eval": prefix_eval,
                }
            )

            dprint("[accept candidate boundary]")
            dprint("[commit sentence memory]")
            dprint("[M_g objects]", global_objects)

            sentence_objects = []
            sentence_masks = []

            if is_eos_stop(current_output, tokenizer):
                break

            continue

        candidates = current_output.get("candidates", None)

        if not candidates:
            # Fallback: accept current output, but keep prefix memory update.
            accepted_generated_ids = current_output["full_generated_ids"]
            processed_prefix_len = len(accepted_generated_ids)

            decision_trace.append(
                {
                    "round": round_idx,
                    "path": "uncertainty_no_candidates_fallback",
                    "stop_reason": current_output["stop_reason"],
                    "prefix_eval": prefix_eval,
                }
            )

            dprint("[fallback accept: no candidates]")

            if is_sentence_boundary_stop(current_output, tokenizer):
                global_objects.extend(sentence_objects)
                global_masks.extend(sentence_masks)

                sentence_objects = []
                sentence_masks = []

                if is_eos_stop(current_output, tokenizer):
                    break

            continue

        dprint("[num_candidates]", len(candidates))

        candidate_outputs = []
        candidate_scores = []
        candidate_evals = []

        # ===================================================
        # 3. Candidate continuation generation + scoring
        # ===================================================
        for cand_rank, cand in enumerate(candidates):
            candidate_id = int(cand["token_id"])
            candidate_text = cand["token_text"]

            dprint(
                f"\n[try candidate {cand_rank}] "
                f"{candidate_id} {repr(candidate_text)}"
            )

            branch_output = generate_sentence_with_attn(
                model=model,
                processor=processor,
                inputs=base_inputs,
                prefix_ids=checkpoint_generated_ids,
                max_new_tokens=max_new_tokens,
                min_new_tokens=min_new_tokens,
                temperature=temperature,
                stop_at_sentence_end=stop_at_sentence_end,
                selected_layers=selected_layers,
                keep_attn_on_cpu=True,
                return_full_attn=False,
                top_k=top_k,
                accumulate_prob=accumulate_prob,
                enable_uncertainty_check=True,
                checkpoint_once=True,

                # The forced candidate should be evaluated first.
                # Candidate-boundary stopping can still occur after that.
                stop_if_sentence_end_in_candidates=True,

                force_first_token_id=candidate_id,
                debug=debug,
            )

            candidate_outputs.append(branch_output)

            continuation_ids = branch_output["generated_ids"]
            continuation_steps = branch_output["steps"]

            continuation_eval = score_object_segment_for_memory(
                tokenizer=tokenizer,
                segment_token_ids=continuation_ids,
                segment_steps=continuation_steps,
                global_objects=global_objects,
                global_masks=global_masks,
                sentence_objects=sentence_objects,
                sentence_masks=sentence_masks,
                image_grid_shape=image_grid_shape,
                inputs=base_inputs,
                iou_thresh=iou_thresh,
                ads_thresh=ads_thresh,
                ads_foreground_ratio=ads_foreground_ratio,
                debug=debug,
            )

            hallucination_score = continuation_eval["hallucination_score"]

            # Section tie-breaker:
            #     argmin (H(C_i), r_i)
            score = (
                hallucination_score,
                cand_rank,
            )

            candidate_scores.append(score)
            candidate_evals.append(continuation_eval)

            dprint("[branch text]", repr(continuation_eval["text"]))
            dprint("[accepted objects]", continuation_eval["accepted_objects"])
            dprint("[suspicious objects]", continuation_eval["suspicious_objects"])
            dprint("[score]", score)

        # ===================================================
        # 4. Candidate reranking
        # ===================================================
        selected_idx = sorted(
            range(len(candidate_scores)),
            key=lambda i: candidate_scores[i],
        )[0]

        selected_output = candidate_outputs[selected_idx]
        selected_eval = candidate_evals[selected_idx]
        selected_candidate = candidates[selected_idx]

        dprint(
            f"\n[select] {repr(selected_candidate['token_text'])} "
            f"score={candidate_scores[selected_idx]}"
        )

        accepted_generated_ids = selected_output["full_generated_ids"]
        processed_prefix_len = len(accepted_generated_ids)

        # Selected grounded objects update M_s.
        sentence_objects.extend(selected_eval["accepted_objects"])
        sentence_masks.extend(selected_eval["accepted_masks"])

        decision_trace.append(
            {
                "round": round_idx,
                "path": "uncertainty_candidate_selection",
                "stop_reason": selected_output["stop_reason"],
                "prefix_eval": prefix_eval,
                "candidate_scores": candidate_scores,
                "selected_idx": selected_idx,
                "selected_candidate": selected_candidate,
                "selected_eval": selected_eval,
            }
        )

        # ---------------------------------------------------
        # 4.1 Commit M_s if selected continuation ends sentence.
        # ---------------------------------------------------
        if is_sentence_boundary_stop(selected_output, tokenizer):
            global_objects.extend(sentence_objects)
            global_masks.extend(sentence_masks)

            dprint("[commit selected sentence memory]")
            dprint("[M_g objects]", global_objects)

            sentence_objects = []
            sentence_masks = []
            processed_prefix_len = len(accepted_generated_ids)

            if is_eos_stop(selected_output, tokenizer):
                break

            continue

        # Otherwise continue same unfinished sentence.
        continue

    final_text = tokenizer.decode(
        accepted_generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "final_text": final_text,
        "final_generated_ids": accepted_generated_ids,

        # Global accepted object memory M_g
        "objects": global_objects,
        "masks": global_masks,
        "global_objects": global_objects,
        "global_masks": global_masks,

        # Remaining unfinished current-sentence memory M_s
        "sentence_objects": sentence_objects,
        "sentence_masks": sentence_masks,

        "processed_prefix_len": processed_prefix_len,
        "round_outputs": round_outputs,
        "decision_trace": decision_trace,
    }

In [29]:
# ============ COCO VAL2017 PHG INFERENCE ============

import json
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm

COCO_IMG_DIR = Path("../data/coco2017/val2017")
COCO_ANN_PATH = Path("../data/coco2017/annotations/instances_val2017.json")

OUTPUT_JSON = "../results/infer/coco_val2017/llava15/greedy_phg.json"

PROMPT = "Describe this image."

MAX_NEW_TOKENS = 256

# For full COCO val2017, set LIMIT = None
# For testing, use LIMIT = 10 or 100
LIMIT = None

# Resume from existing output
RESUME = True

In [30]:
processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def load_image(path):
    return Image.open(path).convert("RGB")


def save_json(rows, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rows = sorted(rows, key=lambda x: x["id"])

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)


def clean_output(text):
    text = str(text).strip()

    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()

    return text.strip()


def extract_caption_from_phg_output(phg_out):
    """
    Robust extractor because different notebook versions may return:
      - string
      - dict with 'text' / 'caption' / 'output_text'
      - list of token ids
    """
    if isinstance(phg_out, str):
        return clean_output(phg_out)

    if isinstance(phg_out, dict):
        for key in ["caption", "text", "output_text", "generated_text", "decoded_text"]:
            if key in phg_out:
                return clean_output(phg_out[key])

        if "generated_ids" in phg_out:
            ids = phg_out["generated_ids"]
            if isinstance(ids, torch.Tensor):
                ids = ids.detach().cpu()
            text = processor.tokenizer.decode(ids, skip_special_tokens=True)
            return clean_output(text)

        if "output_ids" in phg_out:
            ids = phg_out["output_ids"]
            if isinstance(ids, torch.Tensor):
                ids = ids.detach().cpu()
            text = processor.tokenizer.decode(ids, skip_special_tokens=True)
            return clean_output(text)

    if isinstance(phg_out, (list, tuple)):
        # If list of token ids
        if len(phg_out) > 0 and isinstance(phg_out[0], int):
            text = processor.tokenizer.decode(phg_out, skip_special_tokens=True)
            return clean_output(text)

    return clean_output(phg_out)

In [31]:
with open(COCO_ANN_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

images_info = sorted(coco["images"], key=lambda x: int(x["id"]))

if LIMIT is not None:
    images_info = images_info[:LIMIT]

print(f"Loaded {len(images_info)} COCO val2017 images.")
print(images_info[0])

Loaded 5000 COCO val2017 images.
{'license': 2, 'file_name': '000000000139.jpg', 'coco_url': 'http://images.cocodataset.org/val2017/000000000139.jpg', 'height': 426, 'width': 640, 'date_captured': '2013-11-21 01:34:01', 'flickr_url': 'http://farm9.staticflickr.com/8035/8024364858_9c41dc1666_z.jpg', 'id': 139}


In [32]:
output_path = Path(OUTPUT_JSON)

if RESUME and output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    done_ids = {int(row["id"]) for row in results}
    print(f"Resuming from {OUTPUT_JSON}")
    print(f"Already done: {len(done_ids)}")
else:
    results = []
    done_ids = set()

remaining_images = [
    item for item in images_info
    if int(item["id"]) not in done_ids
]

print(f"Remaining: {len(remaining_images)}")

Remaining: 5000


In [ ]:
model.eval()

hf_prompt = f"USER: <image>\n{PROMPT} ASSISTANT:"

for item in tqdm(remaining_images):
    image_id = int(item["id"])
    file_name = item["file_name"]
    image_path = COCO_IMG_DIR / file_name

    if not image_path.exists():
        raise FileNotFoundError(f"Missing image: {image_path}")

    image = load_image(image_path)

    inputs = processor(
        text=hf_prompt,
        images=image,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {
        k: v.to(device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        phg_out = generate_with_phg(
            model=model,
            processor=processor,
            inputs=inputs,
            prefix_ids=None,

            # grounding thresholds
            iou_thresh=0.5,
            ads_thresh=0.45,
            ads_foreground_ratio=0.10,

            # speed-friendly config
            max_rounds=7,
            max_new_tokens=MAX_NEW_TOKENS,
            min_new_tokens=3,
            temperature=0.0,
            stop_at_sentence_end=True,

            # do not use all layers
            selected_layers=None,

            # candidate branching
            top_k=3,
            accumulate_prob=0.5,

            image_grid_shape=(24, 24),
            debug=False,
        )

    caption = phg_out["final_text"].strip()

    results.append({
        "id": image_id,
        "caption": caption,
    })

    print(f"\nCOCO image_id={image_id}")
    print(caption)

    # save after every image because PHG is slow
    save_json(results, OUTPUT_JSON)

print(f"\nSaved {len(results)} captions to {OUTPUT_JSON}")

  0%|          | 1/5000 [02:25<202:13:54, 145.64s/it]


COCO image_id=139
The image shows a spacious and well-lit living room with a dining table in the center. There are four chairs surrounding the table, with two on each side. A woman is standing near the table, possibly preparing to sit down.

The living room is furnished with a TV on the left side, a refrigerator on the right side, and a clock hanging on the wall above the table. There are also two potted plants in the room, one near the left side and the other near the right side.


  0%|          | 2/5000 [03:07<117:47:42, 84.85s/it] 


COCO image_id=285
The image features a large brown bear sitting on a grassy field. The bear is positioned in the center of the scene, occupying a significant portion of the image. The grassy field extends across the background, providing a natural setting for the bear.


  0%|          | 3/5000 [05:45<164:01:35, 118.17s/it]

...


COCO image_id=5586
The image captures a tennis match in progress, with a woman standing on a tennis court holding a tennis racket. She is wearing a yellow shirt and appears to be focused on the game. 

There are several other people in the scene, some of whom are likely spectators or fellow players. A few of them are standing near the edges of the court, while others are scattered around the area. A chair can be seen in the background, possibly for resting or observing the match.


  1%|          | 56/5000 [2:29:57<235:29:23, 171.47s/it]


COCO image_id=5600
The image features a dining table with two bowls filled with food. One bowl is placed towards the left side of the table, while the other is positioned more towards the center. The food in the bowls appears to be a mix of vegetables and meat.

There are several pieces of broccoli scattered around the table, with some placed near the bowls and others spread out across the table. A spoon can be seen resting on the table, likely used for serving the food.


  1%|          | 57/5000 [2:31:55<213:25:07, 155.43s/it]


COCO image_id=5992
The image features a group of sheep standing in a grassy field. There are three sheep in total, with one on the left side, another in the middle, and the third on the right side of the field. They appear to be enjoying their time in the open area.

Additionally, there is a chair located in the background, slightly to the left of the center of the image.
